<a href="https://colab.research.google.com/github/yuzxin/Capstone/blob/main/ASOS_%EB%B3%80%EC%88%98%EC%B6%94%EA%B0%80_%EC%A0%84%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [71]:
import numpy as np
import pandas as pd
from google.colab import drive

In [72]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [73]:
file_path = "/content/drive/MyDrive/캡스톤_오데양/소스데이터셋/사용데이터(원본데이터)/ASOS_ver2.csv"
df = pd.read_csv(file_path, encoding="cp949")

In [74]:
rename_dict = {
    '지점': 'station_id',
    '지점명': 'location',
    '일시': 'date',
    '평균기온(°C)': 'temp_avg',
    '최저기온(°C)': 'temp_min',
    '최고기온(°C)': 'temp_max',
    '일강수량(mm)': 'rainfall',
    '평균 이슬점온도(°C)': 'dew_point_avg',
    '최소 상대습도(%)': 'humidity_min',
    '평균 상대습도(%)': 'humidity_avg',
    '평균 증기압(hPa)': 'vapor_pressure_avg'
}
df = df.rename(columns=rename_dict)

print("=== 원본 데이터 구조 확인 ===")
print(df.info())

=== 원본 데이터 구조 확인 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 348797 entries, 0 to 348796
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   station_id          348797 non-null  int64  
 1   location            348797 non-null  object 
 2   date                348797 non-null  object 
 3   temp_avg            348375 non-null  float64
 4   temp_min            348716 non-null  float64
 5   temp_max            348722 non-null  float64
 6   rainfall            132062 non-null  float64
 7   dew_point_avg       348091 non-null  float64
 8   humidity_min        348587 non-null  float64
 9   humidity_avg        348121 non-null  float64
 10  vapor_pressure_avg  348083 non-null  float64
dtypes: float64(8), int64(1), object(2)
memory usage: 29.3+ MB
None


### 날짜 데이터 형식 변환 (시계열)

In [75]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["location", "date"]).reset_index(drop=True)

# 날짜 파생변수 생성
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_year"] = df["date"].dt.dayofyear

### 결측치 탐지 및 시계열 보정

In [76]:
# 1. 전처리 전 컬럼별 결측치(NaN) 개수 확인
print("--- 전처리 전 빈 데이터 개수 ---")
print(df.isnull().sum())

# 2. 일강수량: 비가 안 온 날(NaN) -> 0.0mm 처리
df["rainfall"] = df["rainfall"].fillna(0.0)

# 3. 기온 및 습도 관련 연속형 변수 7종: 관측지점(location)별 시계열 직전(ffill)/직후(bfill) 값 대치
weather_cols = [
    "temp_avg", "temp_min", "temp_max",
    "dew_point_avg", "humidity_min", "humidity_avg", "vapor_pressure_avg"
]

for col in weather_cols:
    df[col] = df.groupby("location")[col].ffill()
    df[col] = df.groupby("location")[col].bfill()

# 4. 전처리 후 최종 결측치 개수 확인 (모두 0이어야 함)
print("\n--- 전처리 후 빈 데이터 개수 ---")
print(df.isnull().sum())

--- 전처리 전 빈 데이터 개수 ---
station_id                 0
location                   0
date                       0
temp_avg                 422
temp_min                  81
temp_max                  75
rainfall              216735
dew_point_avg            706
humidity_min             210
humidity_avg             676
vapor_pressure_avg       714
year                       0
month                      0
day                        0
day_of_year                0
dtype: int64

--- 전처리 후 빈 데이터 개수 ---
station_id            0
location              0
date                  0
temp_avg              0
temp_min              0
temp_max              0
rainfall              0
dew_point_avg         0
humidity_min          0
humidity_avg          0
vapor_pressure_avg    0
year                  0
month                 0
day                   0
day_of_year           0
dtype: int64


### 데이터 논리 오류 정제 (기온 모순 데이터 이상치 필터링)

In [77]:
# 물리적 이상치 정제 (최저기온 > 최고기온 검증)
invalid_rows = df[df["temp_min"] > df["temp_max"]]
print(f"\n물리적 기온 오류 데이터 개수: {len(invalid_rows)}건")


물리적 기온 오류 데이터 개수: 0건


In [78]:
# 최저기온이 최고기온보다 같거나 낮은 정상적인 데이터만 남기고 필터링
df = df[df["temp_min"] <= df["temp_max"]].reset_index(drop=True)
print(f"이상치 정제 후 남은 총 데이터 개수: {len(df):,}건")

이상치 정제 후 남은 총 데이터 개수: 348,797건


### 분석용 파생 변수 생성 (일교차 및 적산온도)

In [79]:
# 일교차 생성 (최고기온 - 최저기온)
# 일교차 (temp_range)
df["temp_range"] = df["temp_max"] - df["temp_min"]

In [80]:
# 밀원수 개화용 적산온도(GDD) 계산 (기준온도 = 5℃)
BASE_TEMP = 5

# 날짜순 정렬
df = df.sort_values(["location", "date"]).reset_index(drop=True)

# 일별 유효 적산온도 (daily_gdd) 및 연도/지점별 누적 적산온도 (accumulated_temp)
BASE_TEMP = 5
df["daily_gdd"] = np.maximum(0, df["temp_avg"] - BASE_TEMP)
df["accumulated_temp"] = df.groupby(["location", "year"])["daily_gdd"].cumsum()

# 결과 확인
display(
    df[
        [
            "location",
            "date",
            "temp_min",
            "temp_max",
            "daily_gdd",
            "accumulated_temp",
        ]
    ].head()
)

,location,date,temp_min,temp_max,daily_gdd,accumulated_temp
0,강릉,2016-01-01,2.1,8.9,0.1,0.1
1,강릉,2016-01-02,5.5,11.2,4.1,4.2
2,강릉,2016-01-03,5.3,13.8,4.0,8.2
3,강릉,2016-01-04,3.3,12.0,2.4,10.6
4,강릉,2016-01-05,0.4,7.7,0.0,10.6


###아까시나무 개화일 계산식

In [81]:
flower = (
    df[df["accumulated_temp"] >= 200]
      .groupby(["location", "year"])
      .first()
      .reset_index()
)

display(
    flower[
        [
            "location",
            "year",
            "date",
            "accumulated_temp",
        ]
    ]
)

,location,year,date,accumulated_temp
0,강릉,2016,2016-04-09,209.1
1,강릉,2017,2017-04-12,203.4
2,강릉,2018,2018-04-03,211.5
3,강릉,2019,2019-04-12,201.9
4,강릉,2020,2020-04-04,201.9
...,...,...,...,...
952,흑산도,2021,2021-03-27,202.9
953,흑산도,2022,2022-04-16,203.4
954,흑산도,2023,2023-04-11,207.4
955,흑산도,2024,2024-04-05,205.8


In [82]:
df.head()

,station_id,location,date,temp_avg,temp_min,temp_max,rainfall,dew_point_avg,humidity_min,humidity_avg,vapor_pressure_avg,year,month,day,day_of_year,temp_range,daily_gdd,accumulated_temp
0,105,강릉,2016-01-01,5.1,2.1,8.9,0.0,-10.2,20.0,33.0,2.9,2016,1,1,1,6.8,0.1,0.1
1,105,강릉,2016-01-02,9.1,5.5,11.2,0.0,-2.7,35.0,43.8,5.1,2016,1,2,2,5.7,4.1,4.2
2,105,강릉,2016-01-03,9.0,5.3,13.8,0.0,0.4,31.0,55.8,6.3,2016,1,3,3,8.5,4.0,8.2
3,105,강릉,2016-01-04,7.4,3.3,12.0,0.0,-8.6,13.0,33.8,3.5,2016,1,4,4,8.7,2.4,10.6
4,105,강릉,2016-01-05,3.0,0.4,7.7,0.0,-20.6,10.0,16.3,1.2,2016,1,5,5,7.3,0.0,10.6


### 추가 기상 변수 전처리

In [83]:
# 일강수량(rainfall) 결측치 처리 (비가 오지 않은 날 NaN -> 0.0)
df["rainfall"] = df["rainfall"].fillna(0.0)

In [84]:
# 습도 및 기상 관련 변수 결측치 지점별(location) 보정
humidity_cols = ["dew_point_avg", "humidity_min", "humidity_avg", "vapor_pressure_avg"]

for col in humidity_cols:
    df[col] = df.groupby("location")[col].ffill()
    df[col] = df.groupby("location")[col].bfill()

In [85]:
# 전체 전처리 완료 확인 (모든 컬럼의 결측치가 0이어야 함)
print("--- 전처리 완료 데이터 구조 최종 확인 ---")
print(df.info())
print("\n--- 전체 변수 최종 결측치 개수 확인 ---")
print(df.isnull().sum())

--- 전처리 완료 데이터 구조 최종 확인 ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 348797 entries, 0 to 348796
Data columns (total 18 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   station_id          348797 non-null  int64         
 1   location            348797 non-null  object        
 2   date                348797 non-null  datetime64[ns]
 3   temp_avg            348797 non-null  float64       
 4   temp_min            348797 non-null  float64       
 5   temp_max            348797 non-null  float64       
 6   rainfall            348797 non-null  float64       
 7   dew_point_avg       348797 non-null  float64       
 8   humidity_min        348797 non-null  float64       
 9   humidity_avg        348797 non-null  float64       
 10  vapor_pressure_avg  348797 non-null  float64       
 11  year                348797 non-null  int32         
 12  month               348797 non-null  int32         
 13  d

### 정제된 마스터 파일 내보내기

In [86]:
ASOS_변수추가_전처리 = "ASOS_변수추가_전처리.csv"
df.to_csv(ASOS_변수추가_전처리, index=False, encoding="utf-8-sig")

print(f"\n[전처리 완료] 모든 기상 변수가 포함된 최종 파일이 '{ASOS_변수추가_전처리}'(으)로 보관되었습니다.")


[전처리 완료] 모든 기상 변수가 포함된 최종 파일이 'ASOS_변수추가_전처리.csv'(으)로 보관되었습니다.
